### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="blood_tests_drink_prediction",
    dataset_year="1996",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C54G67",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/60/liver+disorders.zip &&  unzip liver+disorders.zip bupa.data && rm liver+disorders.zip
mkdir -p local-data-warehouse/blood_tests_drink_prediction && mv bupa.data local-data-warehouse/blood_tests_drink_prediction/
""",
    # References
    academic_reference_bibtex=r"""@misc{UCILiverDisorders2016,
  title        = {Liver Disorders},
  author       = {{UCI Machine Learning Repository}},
  year         = {2016},
  howpublished = {\url{https://doi.org/10.24432/C54G67}},
  note         = {Dataset}
}
""",
    academic_reference_bibtex_key="",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
The task is framed as a liver disorder prediction task, but we only have data bout the amount of drinks consumed. So instead, we will frame it as a task to predict the number of drinks based on the blood work data. This is a proxy for the original task, where later based on the number of drinks the liver disorder was determined.

- We drop the selector column, as we ignore the original train/test split, and will create our own splits.
- The data contains natural duplicates, so we keep them.
- We log1p scale the target.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="drinks",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv(dataset_mold.path / "bupa.data", header=None, names=["mcv", "alkphos", "sgpt", "sgot", "gammagt", "drinks", "selector"])
print("Loaded data shape:", df.shape)

df = df.drop(columns=["selector"])
df["drinks"] = np.log1p(df["drinks"])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (345, 7)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 345
Columns: 6
Use sampling: False (sample size: 345)
Get row duplicates (staged, merged)...
Using top-5 columns for initial filtering: ['gammagt', 'alkphos', 'sgpt', 'sgot', 'mcv']
Rows remaining as candidates after top-5 filter: 8 (of 345)

#### Duplicate Report
Total duplicate rows: 4 (1.16% of dataset)
Duplicate rows ignoring target: 4 (1.16% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,mcv,alkphos,sgpt,sgot,gammagt,drinks
0,97,62,17,13,5,0.405465
1,92,87,57,25,44,1.945910
2,85,58,18,24,16,0.405465
3,89,82,33,32,18,0.405465
4,95,93,21,27,47,1.945910


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,drinks,float64,0.0,0.0,16.0,"0.4055, 1.6094, 1.9459, 1.0986, 2.1972, 1.3863, 1.7918, 0.6931, 2.3979, 0.0"
1,mcv,int64,0.0,0.0,26.0,"91, 92, 90, 88, 89, 87, 86, 93, 85, 94"
2,alkphos,int64,0.0,0.0,78.0,"63, 62, 67, 57, 55, 58, 65, 80, 70, 60"
3,sgpt,int64,0.0,0.0,67.0,"17, 20, 25, 21, 26, 27, 18, 24, 19, 29"
4,sgot,int64,0.0,0.0,47.0,"20, 23, 21, 26, 19, 22, 25, 18, 17, 24"
5,gammagt,int64,0.0,0.0,94.0,"14, 16, 11, 19, 13, 18, 15, 22, 12, 17"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
mcv,345.0,90.159420,4.448096,65.0,103.000000
alkphos,345.0,69.869565,18.347670,23.0,138.000000
sgpt,345.0,30.405797,19.512309,4.0,155.000000
sgot,345.0,24.643478,10.064494,5.0,82.000000
gammagt,345.0,38.284058,39.254616,5.0,297.000000
drinks,345.0,1.228684,0.737829,0.0,3.044522


In [7]:
# Categorical Feature Statistics
cat_stats

'No categorical/object features to summarize.'

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,2.61,0.114,-0.234,0.544,0.123,log1p,887.9,2717.9,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to blood_tests_drink_prediction/019d5dc0-30db-7725-9eb9-1b2bec176699
019d5dc0-30db-7725-9eb9-1b2bec176699
76c58050bb7b90bbcf2ea71b1c94b455c67ed6acaed566adf728372215c00a9d
